# Classification Tulipes / Lys

Pipeline mono-notebook : parsing -> prétraitement -> entraînement -> visualisation.

Un seul fichier Parquet est écrit, juste avant la visualisation.

> **Règles** : DataFrame uniquement · pas de `collect()` / `toPandas()` / `toList()` · Python pur = affichage seulement


## 0 · Session Spark & imports

In [1]:
import sys
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "\\bin;" + os.environ["PATH"]
import io
import struct

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, FloatType, ArrayType, StringType
)

print(sys.executable)


import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

spark = (
    SparkSession.builder
    .appName("TulipsLilies")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    .config("spark.driver.memory", "2g")        # augmenter
    .config("spark.executor.memory", "2g")       # augmenter
    .config("spark.sql.shuffle.partitions", "4") # réduire les partitions
    .config("spark.python.worker.faulthandler.enabled", "true")
    .getOrCreate()
)

c:\Users\eliet\miniconda3\envs\spark_env\python.exe


## 1 - Chemins & constantes

In [2]:
BASE = os.getcwd()  # c:\Users\eliet\big-data-spark

TRAIN_PATH = os.path.join(BASE, "data", "Train")
TEST_PATH  = os.path.join(BASE, "data", "Test")
OUTPUT_PREDS = os.path.join(BASE, "output", "predictions")
MODEL_PATH   = os.path.join(BASE, "output", "model")
TARGET_SIZE  = (64, 64)

print(TRAIN_PATH)  # vérification

c:\Users\eliet\big-data-spark\data\Train


In [3]:
print(os.environ.get("HADOOP_HOME"))
spark.read.format("binaryFile").load(TRAIN_PATH).limit(1).show()

C:\hadoop
+----+----------------+------+-------+
|path|modificationTime|length|content|
+----+----------------+------+-------+
+----+----------------+------+-------+



## 2 - Parsing

In [1]:
TARGET_W, TARGET_H = TARGET_SIZE

def decode_image_bytes(raw_bytes: bytes):
    try:
        from PIL import Image
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        img = img.resize((TARGET_W, TARGET_H), Image.LANCZOS)
        raw = img.tobytes()
        n = len(raw)
        pixels = list(struct.unpack(f"{n}B", raw))
        return (TARGET_W, TARGET_H, 3, [float(p) for p in pixels])
    except Exception:
        return None

_decode_schema = StructType([
    StructField("width",    IntegerType(), False),
    StructField("height",   IntegerType(), False),
    StructField("channels", IntegerType(), False),
    StructField("pixels",   ArrayType(FloatType()), False),
])

decode_udf = F.udf(decode_image_bytes, _decode_schema)

def parse_images(path):
    raw = (
        spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .load(path)
        .filter(F.lower(F.col("path")).rlike(r"\.(jpg|jpeg|png)$"))  # ← filtre après
    )
    return (
        raw
        .select(
            F.regexp_extract(F.col("path"), r"([^/\\]+)$", 1).alias("image_id"),
            F.regexp_extract(F.col("path"), r"[/\\]([^/\\]+)[/\\][^/\\]+$", 1).alias("label"),
            F.col("content").alias("raw_bytes"),
        )
        .withColumn("decoded", decode_udf(F.col("raw_bytes")))
        .filter(F.col("decoded").isNotNull())
        .select(
            "image_id", "label",
            F.col("decoded.pixels").alias("pixels"),
        )
    )

train_parsed_df = parse_images(TRAIN_PATH)
test_parsed_df  = parse_images(TEST_PATH)

print(f"Images train : {train_parsed_df.count()}")
print(f"Images test  : {test_parsed_df.count()}")
test_parsed_df.show()

NameError: name 'TARGET_SIZE' is not defined

In [5]:
def get_error(raw_bytes: bytes):
    try:
        from PIL import Image
        import io
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        img = img.resize((64, 64), Image.LANCZOS)
        return "OK"
    except Exception as e:
        return str(e)

error_udf = F.udf(get_error, StringType())

(
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(TRAIN_PATH)
    .filter(F.lower(F.col("path")).rlike(r"\.(jpg|jpeg|png)$"))
    .limit(1)
    .withColumn("erreur", error_udf(F.col("content")))
    .select("path", "erreur")
    .show(truncate=False)
)

+-----------------------------------------------------------------+------+
|path                                                             |erreur|
+-----------------------------------------------------------------+------+
|file:/c:/Users/eliet/big-data-spark/data/Train/tulipes/000055.png|OK    |
+-----------------------------------------------------------------+------+



## 3 - Prétraitement 

On garde `pixels` (RGB brut, 0-255) intact pour l'affichage futur dans Streamlit,
et on ajoute une colonne `pixels_gray` : niveaux de gris normalisés en [0.0, 1.0].

Conversion RGB -> nuances de gris : formule de luminance pondérée (standard) :
```
gray = 0.299*R + 0.587*G + 0.114*B
```

`pixels` est une liste aplatie `[R,G,B, R,G,B, ...]` de taille 64*64*3 = 12288.
L'UDF regroupe les valeurs par 3, applique la formule et renvoie les pixels en niveaux de gris

In [6]:
def rgb_to_grayscale(pixels):
    if pixels is None:
        return None
    gray = []
    for i in range(0, len(pixels), 3):
        r, g, b = pixels[i], pixels[i + 1], pixels[i + 2]
        # formule standard de luminance perceptuelle
        g_value = 0.299 * r + 0.587 * g + 0.114 * b
        gray.append(g_value)
    return gray

gray_udf = F.udf(rgb_to_grayscale, ArrayType(FloatType()))

def preprocess(df):
    return df.withColumn("pixels_grayscale", gray_udf(F.col("pixels")))

train_preprocessed_gray_df = preprocess(train_parsed_df)
test_preprocessed_gray_df  = preprocess(test_parsed_df)

print("Aperçu après prétraitement :")
train_preprocessed_gray_df.select("image_id", "label", "pixels_grayscale").show(5, truncate=40)

# Vérification rapide : taille attendue = 64*64 = 4096 valeurs en niveaux de gris
expected_len = TARGET_W * TARGET_H
check_len = (
    train_preprocessed_gray_df
    .select(F.size(F.col("pixels_grayscale")).alias("len"))
    .first()["len"]
)
print(f"Taille pixels_grayscale (attendu {expected_len}) : {check_len}")

Aperçu après prétraitement :
+----------+-------+----------------------------------------+
|  image_id|  label|                        pixels_grayscale|
+----------+-------+----------------------------------------+
|000055.png|tulipes|[183.901, 197.972, 194.972, 194.26, 1...|
|000200.jpg|tulipes|[165.793, 167.369, 168.657, 169.059, ...|
|000162.jpg|tulipes|[163.244, 149.891, 130.483, 136.26, 1...|
|000069.png|    lys|[47.279, 47.301, 52.957, 58.013, 54.1...|
|000098.jpg|tulipes|[144.913, 150.169, 156.038, 156.152, ...|
+----------+-------+----------------------------------------+
only showing top 5 rows

Taille pixels_grayscale (attendu 4096) : 4096


In [7]:
def rgb_to_normalized_gray(pixels):
    if pixels is None:
        return None
    gray = []
    for i in range(0, len(pixels), 3):
        r, g, b = pixels[i], pixels[i + 1], pixels[i + 2]
        g_value = 0.299 * r + 0.587 * g + 0.114 * b
        gray.append(g_value / 255.0)
    return gray

gray_norm_udf = F.udf(rgb_to_normalized_gray, ArrayType(FloatType()))

def preprocess_norm(df):
    return df.withColumn("pixels_gray", gray_norm_udf(F.col("pixels")))

train_preprocessed_norm_gray_df = preprocess_norm(train_parsed_df)
test_preprocessed_norm_gray_df  = preprocess_norm(test_parsed_df)

print("Aperçu :")
train_preprocessed_norm_gray_df.select("image_id", "label", "pixels_gray").show(5, truncate=40)

Aperçu :


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "c:\Users\eliet\miniconda3\envs\spark_env\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\eliet\miniconda3\envs\spark_env\Lib\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\eliet\miniconda3\envs\spark_env\Lib\socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

## ML couleurs - prétraitement en bytes

## ML couleurs normalisées


## ML grayscale

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import numpy as np

LABEL_COL   = "label"
FEATURE_COL = "pixels_gray"           # ← pas pixels_norm

def to_numpy(df, feature_col=FEATURE_COL, label_col=LABEL_COL):  # ← à définir dans la cellule
    rows = df.select("image_id", label_col, feature_col).collect()
    image_ids = [r["image_id"] for r in rows]
    X = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y = [r[label_col] for r in rows]
    return image_ids, X, y

train_ids, X_train_gray, y_train_raw = to_numpy(train_preprocessed_norm_gray_df)  # ← bon nom
test_ids,  X_test_gray,  y_test_raw  = to_numpy(test_preprocessed_norm_gray_df)   # ← bon nom

label_encoder = LabelEncoder()          # ← à définir ici, pas réutiliser depuis ailleurs
label_encoder.fit(y_train_raw)
y_train = label_encoder.transform(y_train_raw)
y_test  = label_encoder.transform(y_test_raw)
print(f"Classes : {dict(enumerate(label_encoder.classes_))}")

# --- Entraînement RandomForest ---
rf_gray_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
rf_gray_model.fit(X_train_gray, y_train)

# --- Évaluation ---
y_pred_gray       = rf_gray_model.predict(X_test_gray)
y_pred_proba_gray = rf_gray_model.predict_proba(X_test_gray)

acc_gray = accuracy_score(y_test, y_pred_gray)
print(f"\nAccuracy (grayscale normalisé) : {acc_gray:.4f}")
print(classification_report(y_test, y_pred_gray, target_names=label_encoder.classes_))

# --- Remontée en Spark DataFrame ---
predicted_labels_gray = label_encoder.inverse_transform(y_pred_gray)
confidences_gray      = y_pred_proba_gray.max(axis=1).tolist()

predictions_gray_df = spark.createDataFrame(
    list(zip(test_ids, y_test_raw, predicted_labels_gray.tolist(), confidences_gray)),
    schema=StructType([
        StructField("image_id",        StringType(), False),
        StructField("true_label",      StringType(), False),
        StructField("predicted_label", StringType(), False),
        StructField("confidence",      FloatType(),  False),
    ]),
)

print("\nPrédictions grayscale (Spark DataFrame) :")
predictions_gray_df.show(10, truncate=False)

NameError: name 'train_preprocessed_norm_gray_df' is not defined

## ML grayscale normalisé

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import numpy as np

LABEL_COL   = "label"
FEATURE_COL = "pixels_gray"

def to_numpy(df, feature_col=FEATURE_COL, label_col=LABEL_COL):
    rows = df.select("image_id", label_col, feature_col).collect()
    image_ids = [r["image_id"] for r in rows]
    X = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y = [r[label_col] for r in rows]
    return image_ids, X, y

train_ids, X_train_gray, y_train_raw = to_numpy(train_preprocessed_norm_gray_df)
test_ids,  X_test_gray,  y_test_raw  = to_numpy(test_preprocessed_norm_gray_df)

print(f"X_train_gray shape : {X_train_gray.shape}")
print(f"X_test_gray  shape : {X_test_gray.shape}")

label_encoder = LabelEncoder()
label_encoder.fit(y_train_raw)
y_train = label_encoder.transform(y_train_raw)
y_test  = label_encoder.transform(y_test_raw)
print(f"Classes : {dict(enumerate(label_encoder.classes_))}")

rf_gray_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
rf_gray_model.fit(X_train_gray, y_train)

y_pred_gray       = rf_gray_model.predict(X_test_gray)
y_pred_proba_gray = rf_gray_model.predict_proba(X_test_gray)

acc_gray = accuracy_score(y_test, y_pred_gray)
print(f"\nAccuracy (grayscale normalisé) : {acc_gray:.4f}")
print(classification_report(y_test, y_pred_gray, target_names=label_encoder.classes_))

predicted_labels_gray = label_encoder.inverse_transform(y_pred_gray)
confidences_gray      = y_pred_proba_gray.max(axis=1).tolist()

predictions_gray_df = spark.createDataFrame(
    list(zip(test_ids, y_test_raw, predicted_labels_gray.tolist(), confidences_gray)),
    schema=StructType([
        StructField("image_id",        StringType(), False),
        StructField("true_label",      StringType(), False),
        StructField("predicted_label", StringType(), False),
        StructField("confidence",      FloatType(),  False),
    ]),
)

print("\nPrédictions grayscale (Spark DataFrame) :")
predictions_gray_df.show(10, truncate=False)

X_train_gray shape : (10, 4096)
X_test_gray  shape : (10, 4096)
Classes : {0: np.str_('lys'), 1: np.str_('tulipes')}

Accuracy (grayscale normalisé) : 0.7000
              precision    recall  f1-score   support

         lys       0.75      0.60      0.67         5
     tulipes       0.67      0.80      0.73         5

    accuracy                           0.70        10
   macro avg       0.71      0.70      0.70        10
weighted avg       0.71      0.70      0.70        10


Prédictions grayscale (Spark DataFrame) :
+----------+----------+---------------+----------+
|image_id  |true_label|predicted_label|confidence|
+----------+----------+---------------+----------+
|000139.jpg|tulipes   |tulipes        |0.58      |
|000137.jpg|tulipes   |tulipes        |0.535     |
|000063.jpg|lys       |tulipes        |0.615     |
|000140.jpg|tulipes   |lys            |0.6       |
|000061.jpg|lys       |tulipes        |0.715     |
|000138.jpg|tulipes   |tulipes        |0.535     |
|000062.jpg|l